<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/14-trees-and-ensembles/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 14: Exercises

Twelve exercises: four Easy, five Medium and three Hard. You'll build tree models from their parts: impurity measures, prediction with a stored tree, the split search (for classes and, fast, for numbers), bagging with out-of-bag predictions, permutation importance and partial dependence, and finally a complete decision tree and gradient boosting for regression and for classification.

Several checks compare your code with scikit-learn's trees and boosting models, so when they pass, you know your version follows the same algorithm.

**Run the setup cell first.**

Rules:

- NumPy and pandas. Where an exercise says to use `DecisionTreeRegressor` as a building block, do; otherwise don't call scikit-learn's version of what you're writing.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import partial_dependence
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _gini_ref(labels):
    _, counts = np.unique(np.asarray(labels), return_counts=True)
    p = counts / counts.sum()
    return float(1 - np.sum(p ** 2))


def _weighted_gini_ref(left, right):
    left, right = np.asarray(left), np.asarray(right)
    return (len(left) * _gini_ref(left) + len(right) * _gini_ref(right)) / (len(left) + len(right))


def _predict_ref(tree, X):
    out = []
    for row in np.asarray(X, dtype=float):
        node = tree
        while "value" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        out.append(node["value"])
    return np.array(out)


def check_gini_impurity(f):
    rng = np.random.default_rng(0)
    labels = rng.choice(["a", "b", "c"], 500, p=[0.5, 0.3, 0.2])
    p = pd.Series(labels).value_counts(normalize=True).to_numpy()
    _scenarios("gini_impurity", [
        ("a pure group is 0", lambda: f(["a", "a", "a"]), 0.0, None),
        ("two equal classes give 0.5", lambda: f([0, 1, 0, 1]), 0.5, "1 - (0.5² + 0.5²)"),
        ("three equal classes give 2/3", lambda: f(["x", "y", "z"]), 2 / 3, None),
        ("a 90/10 group", lambda: f([1] * 9 + [0]), 1 - 0.81 - 0.01, None),
        ("random labels", lambda: f(labels), float(1 - np.sum(p ** 2)), None),
        ("bools and arrays work", lambda: f(np.array([True, False, True, True])), 1 - 0.75 ** 2 - 0.25 ** 2, None),
        ("returns a float", lambda: type(f([1, 2])).__name__, "float", None),
        ("an empty group raises", lambda: f([]), Raises(ValueError), None),
    ])


def check_entropy(f):
    _scenarios("entropy", [
        ("a pure group is exactly 0.0", lambda: (f(["a"] * 5), math.copysign(1, f(["a"] * 5))), (0.0, 1.0),
         "-0.0 is not allowed: write it as Σ p log₂(1/p)"),
        ("two equal classes give 1 bit", lambda: f([1, 1, 0, 0]), 1.0, None),
        ("four equal classes give 2 bits", lambda: f(["a", "b", "c", "d"]), 2.0, "use log base 2"),
        ("a 90/10 group", lambda: f([1] * 9 + [0]), -(0.9 * math.log2(0.9) + 0.1 * math.log2(0.1)), None),
        ("matches scipy", lambda: f([0, 0, 1, 2, 2, 2]), float(scipy_entropy([2, 1, 3], base=2)), None),
        ("returns a float", lambda: type(f([1, 2])).__name__, "float", None),
        ("an empty group raises", lambda: f([]), Raises(ValueError), None),
    ])


def check_weighted_impurity(f):
    rng = np.random.default_rng(1)
    y = rng.integers(0, 3, 200)

    def never_worse_than_parent():
        parent = 1 - np.sum((np.bincount(y) / len(y)) ** 2)
        for _ in range(50):
            mask = rng.random(200) < rng.random()
            if f(y[mask], y[~mask]) > parent + 1e-12:
                return False
        return True

    _scenarios("weighted_impurity", [
        ("a perfect split gives 0", lambda: f([0, 0, 0], [1, 1]), 0.0, None),
        ("weights by size", lambda: f([0, 1], [0, 0, 0, 0, 0, 0, 0, 1]), (2 * 0.5 + 8 * (1 - 0.875 ** 2 - 0.125 ** 2)) / 10,
         "(n_left · I(left) + n_right · I(right)) / n"),
        ("entropy criterion", lambda: f([0, 1], [1, 1, 1, 1], criterion="entropy"), 2 / 6 * 1.0, None),
        ("an empty side contributes nothing", lambda: f([], [0, 1]), 0.5, None),
        ("never worse than the parent", never_worse_than_parent, True, None),
        ("returns a float", lambda: type(f([0], [1])).__name__, "float", None),
        ("an unknown criterion raises", lambda: f([0], [1], criterion="mse"), Raises(ValueError), None),
        ("two empty sides raise", lambda: f([], []), Raises(ValueError), None),
    ])


def check_predict_with_tree(f):
    example = {"feature": 0, "threshold": 2.5,
               "left": {"value": "small"},
               "right": {"feature": 1, "threshold": 10, "left": {"value": "medium"}, "right": {"value": "large"}}}
    rng = np.random.default_rng(2)
    X = rng.normal(size=(200, 3))
    y = (X[:, 0] + X[:, 1] * X[:, 2] > 0).astype(int)
    sk = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X, y)

    def to_dict(node=0):
        t = sk.tree_
        if t.children_left[node] == -1:
            return {"value": int(sk.classes_[np.argmax(t.value[node][0])])}
        return {"feature": int(t.feature[node]), "threshold": float(t.threshold[node]),
                "left": to_dict(t.children_left[node]), "right": to_dict(t.children_right[node])}

    X_new = rng.normal(size=(300, 3))
    _scenarios("predict_with_tree", [
        ("the example", lambda: f(example, [[1, 50], [3, 5], [3, 20]]).tolist(), ["small", "medium", "large"],
         "go left when x[feature] <= threshold"),
        ("a value equal to the threshold goes left", lambda: f(example, [[2.5, 99]]).tolist(), ["small"], "use <=, not <"),
        ("a single leaf", lambda: f({"value": 7}, [[1, 2], [3, 4]]).tolist(), [7, 7], None),
        ("a DataFrame works", lambda: f(example, pd.DataFrame({"a": [0, 4], "b": [0, 0]})).tolist(), ["small", "medium"], None),
        ("returns an array", lambda: type(f(example, [[0, 0]])).__name__, "ndarray", None),
        ("matches a scikit-learn tree", lambda: f(to_dict(), X_new).tolist(), sk.predict(X_new).tolist(), None),
    ])


def check_best_split(f):
    rng = np.random.default_rng(3)
    x = np.round(rng.normal(size=400), 3)
    y = np.where(x + rng.normal(0, 0.7, 400) > 0.4, "yes", "no")
    sk = DecisionTreeClassifier(max_depth=1, random_state=0).fit(x[:, None], y)
    x_pos = np.exp(x)
    _scenarios("best_split", [
        ("a small example", lambda: f([1, 2, 3, 4], [0, 0, 1, 1]), (2.5, 0.0), "thresholds are midpoints between distinct values"),
        ("ties go to the smallest threshold", lambda: f([1, 2, 3], [0, 1, 0]), (1.5, 1 / 3), None),
        ("repeated x values", lambda: f([1, 1, 2, 2, 3], ["a", "a", "b", "a", "b"]),
         (1.5, _weighted_gini_ref(["a", "a"], ["b", "a", "b"])), None),
        ("matches scikit-learn's root split", lambda: (round(f(x, y)[0], 6), round(f(x, y)[1], 6)),
         (round(float(sk.tree_.threshold[0]), 6), round(_weighted_gini_ref(y[x <= sk.tree_.threshold[0]], y[x > sk.tree_.threshold[0]]), 6)), None),
        ("only the order of x matters", lambda: np.sum(x_pos <= f(x_pos, y)[0]), int(np.sum(x <= sk.tree_.threshold[0])), None),
        ("one distinct value: no split", lambda: f([5, 5, 5], [0, 1, 1]), (None, 1 - (1 / 3) ** 2 - (2 / 3) ** 2), None),
    ])


def check_best_split_regression(f):
    rng = np.random.default_rng(4)
    x = rng.uniform(0, 10, 20_000)
    y = np.where(x < 6.2, 1.0, 3.0) + np.sin(x) + rng.normal(0, 0.5, 20_000)
    sk = DecisionTreeRegressor(max_depth=1, random_state=0).fit(x[:, None], y)
    x_small = np.round(rng.uniform(0, 1, 300), 2)
    y_small = 2 * x_small + rng.normal(0, 0.2, 300)

    def brute(xv, yv):
        best = (None, np.inf)
        values = np.unique(xv)
        for t in (values[:-1] + values[1:]) / 2:
            left, right = yv[xv <= t], yv[xv > t]
            mse = (np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2)) / len(yv)
            if mse < best[1] - 1e-12:
                best = (float(t), float(mse))
        return best

    _scenarios("best_split_regression", [
        ("a small example", lambda: f([1, 2, 3, 4], [5.0, 5.0, 9.0, 9.0]), (2.5, 0.0), None),
        ("repeated x values", lambda: f([1, 1, 2, 3], [0.0, 2.0, 10.0, 10.0]), (1.5, 0.5), "never split between equal x values"),
        ("ties go to the smallest threshold", lambda: f([1, 2, 3], [0.0, 5.0, 0.0]), (1.5, 12.5 / 3), None),
        ("matches brute force with many repeated values", lambda: f(x_small, y_small), brute(x_small, y_small), None),
        ("20,000 points: matches scikit-learn's threshold", lambda: round(f(x, y)[0], 4), round(float(sk.tree_.threshold[0]), 4),
         "sort once, then use cumulative sums"),
        ("one distinct value: no split", lambda: f([2, 2, 2], [1.0, 2.0, 3.0]), (None, 2 / 3), None),
    ])


def check_bagging_with_oob(f):
    rng = np.random.default_rng(5)
    X = rng.uniform(-2, 2, size=(300, 3))
    y = X[:, 0] ** 2 + np.sin(2 * X[:, 1]) + rng.normal(0, 0.3, 300)
    X_new = rng.uniform(-2, 2, size=(2000, 3))
    y_new = X_new[:, 0] ** 2 + np.sin(2 * X_new[:, 1]) + rng.normal(0, 0.3, 2000)

    def reference(n_trees, seed=0, Xn=X_new):
        g = np.random.default_rng(seed)
        new_total, oob_total, oob_count = np.zeros(len(Xn)), np.zeros(len(y)), np.zeros(len(y))
        for _ in range(n_trees):
            idx = g.integers(0, len(y), len(y))
            tree = DecisionTreeRegressor(random_state=0).fit(X[idx], y[idx])
            new_total += tree.predict(Xn)
            oob = np.ones(len(y), dtype=bool)
            oob[idx] = False
            oob_total[oob] += tree.predict(X[oob])
            oob_count[oob] += 1
        with np.errstate(invalid="ignore", divide="ignore"):
            return new_total / n_trees, np.where(oob_count > 0, oob_total / oob_count, np.nan)

    ref = reference(50)
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, y, X_new, 50)
        return result["v"]

    def oob_close_to_test():
        new, oob = run()
        oob_rmse = np.sqrt(np.nanmean((y - oob) ** 2))
        test_rmse = np.sqrt(np.mean((y_new - new) ** 2))
        return bool(abs(oob_rmse - test_rmse) / test_rmse < 0.15)

    _scenarios("bagging_with_oob", [
        ("predictions for new data match the reference", lambda: run()[0], ref[0],
         "idx = rng.integers(0, n, n) for each tree, from one rng"),
        ("out-of-bag predictions match the reference", lambda: np.nan_to_num(run()[1], nan=-999.0), np.nan_to_num(ref[1], nan=-999.0), None),
        ("the seed is used", lambda: f(X, y, X_new[:5], 10, seed=3)[0], reference(10, seed=3, Xn=X_new[:5])[0], None),
        ("with 2 trees, rows seen by both are nan", lambda: int(np.isnan(f(X, y, X_new[:5], 2)[1]).sum()),
         int(np.isnan(reference(2, Xn=X_new[:5])[1]).sum()), "use np.nan when a row was never out of bag"),
        ("the OOB error is close to the test error", oob_close_to_test, True, None),
    ])


def check_permutation_importance_scores(f):
    rng = np.random.default_rng(6)
    X = pd.DataFrame({"strong": rng.normal(size=600), "weak": rng.normal(size=600), "noise": rng.normal(size=600)})
    y = 3 * X["strong"] + 0.7 * X["weak"] + rng.normal(0, 0.5, 600)
    model = LinearRegression().fit(X[:400], y[:400])
    X_test, y_test = X[400:].reset_index(drop=True), y[400:].reset_index(drop=True)

    def reference(n_repeats=5, seed=0):
        baseline = mean_squared_error(y_test, model.predict(X_test))
        g = np.random.default_rng(seed)
        out = {}
        for column in X_test.columns:
            values = []
            for _ in range(n_repeats):
                Xs = X_test.copy()
                Xs[column] = g.permutation(Xs[column].to_numpy())
                values.append(mean_squared_error(y_test, model.predict(Xs)) - baseline)
            out[column] = float(np.mean(values))
        return pd.Series(out)

    before = X_test.copy()

    def unchanged():
        f(model, X_test, y_test, mean_squared_error)
        return bool(X_test.equals(before))

    _scenarios("permutation_importance_scores", [
        ("matches the reference procedure", lambda: f(model, X_test, y_test, mean_squared_error), reference(),
         "one rng; columns in order; n_repeats shuffles each"),
        ("the seed and repeats are used", lambda: f(model, X_test, y_test, mean_squared_error, n_repeats=3, seed=9), reference(3, 9), None),
        ("ranks strong > weak > noise", lambda: list(f(model, X_test, y_test, mean_squared_error).sort_values(ascending=False).index),
         ["strong", "weak", "noise"], None),
        ("X is not changed", unchanged, True, "shuffle a copy"),
    ])


def check_partial_dependence_1d(f):
    rng = np.random.default_rng(7)
    X = pd.DataFrame({"a": rng.uniform(0, 10, 400), "b": rng.normal(size=400), "c": rng.integers(0, 5, 400).astype(float)})
    y = np.sin(X["a"]) + X["b"] * (X["c"] > 2) + rng.normal(0, 0.2, 400)
    forest = RandomForestRegressor(n_estimators=30, random_state=0).fit(X, y)
    linear = LinearRegression().fit(X, y)
    ref = partial_dependence(forest, X, ["a"], method="brute", grid_resolution=15)
    grid = ref["grid_values"][0]
    before = X.copy()

    def unchanged():
        f(forest, X, "b", [0.0, 1.0])
        return bool(X.equals(before))

    _scenarios("partial_dependence_1d", [
        ("matches scikit-learn for a forest", lambda: f(forest, X, "a", grid), ref["average"][0], None),
        ("a linear model gives a straight line with the weight as slope",
         lambda: np.diff(f(linear, X, "b", [0.0, 1.0, 2.0])), np.full(2, linear.coef_[1]), None),
        ("returns floats", lambda: f(forest, X, "c", [1, 2]).dtype.kind, "f", None),
        ("X is not changed", unchanged, True, "work on a copy"),
    ])


def check_build_tree(f):
    rng = np.random.default_rng(8)
    X = rng.normal(size=(250, 4))
    y = ((X[:, 0] + X[:, 1] ** 2 + rng.normal(0, 0.5, 250)) > 1).astype(int)
    X3 = rng.normal(size=(240, 3))
    y3 = np.select([X3[:, 0] > 0.5, X3[:, 1] > 0], ["a", "b"], "c")
    y3 = np.where(rng.random(240) < 0.1, "c", y3)

    def compare(Xd, yd, depth, leaf):
        sk = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=0).fit(Xd, yd)
        tree = f(Xd, yd, depth, leaf)
        return bool((_predict_ref(tree, Xd) == sk.predict(Xd)).all())

    def depth_of(node):
        return 0 if "value" in node else 1 + max(depth_of(node["left"]), depth_of(node["right"]))

    def smallest_leaf(node, rows):
        if "value" in node:
            return len(rows)
        go_left = X[rows, node["feature"]] <= node["threshold"]
        return min(smallest_leaf(node["left"], rows[go_left]), smallest_leaf(node["right"], rows[~go_left]))

    _scenarios("build_tree", [
        ("a stump on a small example", lambda: f([[1], [2], [3], [4]], [0, 0, 1, 1], max_depth=1),
         {"feature": 0, "threshold": 2.5, "left": {"value": 0}, "right": {"value": 1}}, None),
        ("a pure node is a leaf", lambda: f([[1], [2]], [1, 1], max_depth=3), {"value": 1}, None),
        ("leaf values are plain Python values", lambda: type(f([[1], [2]], [1, 1], max_depth=3)["value"]).__name__, "int",
         "use .item() on the NumPy value"),
        ("ties in a leaf go to the smallest label", lambda: f([[1], [1]], ["b", "a"], max_depth=2), {"value": "a"}, None),
        ("depth 2 matches scikit-learn", lambda: compare(X, y, 2, 1), True, None),
        ("depth 4 matches scikit-learn", lambda: compare(X, y, 4, 1), True, None),
        ("min_samples_leaf=10 matches scikit-learn", lambda: compare(X, y, 5, 10), True, "both sides need at least min_samples_leaf rows"),
        ("three string classes match scikit-learn", lambda: compare(X3, y3, 3, 5), True, None),
        ("max_depth is respected", lambda: depth_of(f(X, y, 3)) <= 3, True, None),
        ("min_samples_leaf is respected", lambda: smallest_leaf(f(X, y, 6, 15), np.arange(len(y))) >= 15, True, None),
    ])


def check_gradient_boosting_regressor(f):
    rng = np.random.default_rng(9)
    X = rng.uniform(-2, 2, size=(400, 3))
    y = X[:, 0] ** 2 + np.sin(2 * X[:, 1]) * X[:, 2] + rng.normal(0, 0.2, 400)
    X_new = rng.uniform(-2, 2, size=(1000, 3))
    y_new = X_new[:, 0] ** 2 + np.sin(2 * X_new[:, 1]) * X_new[:, 2] + rng.normal(0, 0.2, 1000)

    def reference(n, rate, depth, Xn):
        start = y.mean()
        current = np.full(len(y), start)
        out = np.full(len(Xn), start)
        for _ in range(n):
            tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y - current)
            current += rate * tree.predict(X)
            out += rate * tree.predict(Xn)
        return out

    def training_error_falls():
        errors = [np.mean((y - f(X, y, n, 0.1, 2)(X)) ** 2) for n in [1, 5, 20, 60]]
        return bool(all(a > b for a, b in zip(errors, errors[1:])))

    def close_to_sklearn():
        sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
        ours = np.sqrt(np.mean((y_new - f(X, y, 100, 0.1, 3)(X_new)) ** 2))
        theirs = np.sqrt(np.mean((y_new - sk.predict(X_new)) ** 2))
        return bool(abs(ours - theirs) / theirs < 0.1)

    _scenarios("gradient_boosting_regressor", [
        ("zero trees predict the mean", lambda: f(X, y, 0, 0.1, 2)(X_new[:3]), np.full(3, y.mean()), "start from y.mean()"),
        ("matches the reference", lambda: f(X, y, 50, 0.1, 3)(X_new), reference(50, 0.1, 3, X_new),
         "fit each tree to y - current predictions, and add learning_rate × its predictions"),
        ("another setting", lambda: f(X, y, 20, 0.5, 1)(X_new), reference(20, 0.5, 1, X_new), None),
        ("the training error falls with more trees", training_error_falls, True, None),
        ("learning rate 1 with unlimited depth memorises at once", lambda: float(np.max(np.abs(y - f(X, y, 1, 1.0, None)(X)))) < 1e-8,
         True, None),
        ("close to scikit-learn's GradientBoostingRegressor", close_to_sklearn, True, None),
    ])


def check_binary_gradient_boosting(f):
    rng = np.random.default_rng(10)
    X = rng.normal(size=(500, 3))
    y = (rng.random(500) < 1 / (1 + np.exp(-(1.5 * X[:, 0] - X[:, 1] ** 2 + 0.5)))).astype(int)
    X_new = rng.normal(size=(800, 3))

    def reference(n, rate, depth, Xn):
        rate_1 = y.mean()
        start = np.log(rate_1 / (1 - rate_1))
        F, F_new = np.full(len(y), start), np.full(len(Xn), start)
        for _ in range(n):
            p = 1 / (1 + np.exp(-F))
            r = y - p
            tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, r)
            leaves = tree.apply(X)
            values = {leaf: r[leaves == leaf].sum() / (p[leaves == leaf] * (1 - p[leaves == leaf])).sum() for leaf in np.unique(leaves)}
            F += rate * np.array([values[leaf] for leaf in leaves])
            F_new += rate * np.array([values[leaf] for leaf in tree.apply(Xn)])
        return 1 / (1 + np.exp(-F_new))

    def loss_falls():
        losses = [log_loss(y, f(X, y, n, 0.1, 2)(X)) for n in [1, 5, 20, 60]]
        return bool(all(a > b for a, b in zip(losses, losses[1:])))

    def close_to_sklearn():
        sk = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
        return bool(np.max(np.abs(f(X, y, 50, 0.1, 3)(X_new) - sk.predict_proba(X_new)[:, 1])) < 0.1)

    _scenarios("binary_gradient_boosting", [
        ("zero trees predict the base rate", lambda: f(X, y, 0, 0.1, 2)(X_new[:3]), np.full(3, y.mean()), "start from the log-odds of the base rate"),
        ("matches the reference", lambda: f(X, y, 30, 0.1, 3)(X_new), reference(30, 0.1, 3, X_new),
         "leaf value = Σ r / Σ p(1 - p) over the training examples in the leaf"),
        ("another setting", lambda: f(X, y, 10, 0.5, 1)(X_new), reference(10, 0.5, 1, X_new), None),
        ("probabilities are between 0 and 1", lambda: bool(((lambda p: (p > 0) & (p < 1))(f(X, y, 20, 0.3, 3)(X_new))).all()), True, None),
        ("the training log loss falls", loss_falls, True, None),
        ("close to scikit-learn's GradientBoostingClassifier", close_to_sklearn, True, None),
        ("one class only raises", lambda: f(X, np.ones(500), 5, 0.1, 2), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `gini_impurity`

Write `gini_impurity(labels)` that returns the Gini impurity of a group of class labels as a float:

$$G = 1 - \sum_k p_k^2$$

where `p_k` is the fraction of the labels that belong to class `k`.

- Labels can be any values (ints, strings, bools), in a list or an array.
- Raise `ValueError` for an empty group.

Examples: `gini_impurity(["a", "a", "a"])` is `0.0`, and `gini_impurity([0, 1, 0, 1])` is `0.5`.

In [2]:
def gini_impurity(labels) -> float:
    # your code here
    pass

In [3]:
check_gini_impurity(gini_impurity)

FAIL  gini_impurity: a pure group is 0
      expected: 0.0
      got:      None


### Exercise 2: `entropy`

Write `entropy(labels)` that returns the entropy of a group of class labels, in bits, as a float:

$$H = -\sum_k p_k \log_2 p_k$$

- Classes that don't appear contribute nothing (they don't appear in the sum at all).
- A pure group must give exactly `0.0`, not `-0.0`.
- Raise `ValueError` for an empty group.

Examples: `entropy([1, 1, 0, 0])` is `1.0`, and `entropy(["a", "b", "c", "d"])` is `2.0`.

In [4]:
def entropy(labels) -> float:
    # your code here
    pass

In [5]:
check_entropy(entropy)

FAIL  entropy: a pure group is exactly 0.0
      raised TypeError: must be real number, not NoneType
      expected: (0.0, 1.0)
      hint: -0.0 is not allowed: write it as Σ p log₂(1/p)


### Exercise 3: `weighted_impurity`

Write `weighted_impurity(y_left, y_right, criterion="gini")` that scores a split: the impurity of the two groups, averaged with weights equal to their sizes.

$$\frac{n_L \cdot I(\text{left}) + n_R \cdot I(\text{right})}{n_L + n_R}$$

- `criterion` is `"gini"` or `"entropy"`; raise `ValueError` for anything else.
- One side may be empty (it contributes nothing). Raise `ValueError` if both are empty.

Your `gini_impurity` and `entropy` from Exercises 1 and 2 are available.

In [6]:
def weighted_impurity(y_left, y_right, criterion: str = "gini") -> float:
    # your code here
    pass

In [7]:
check_weighted_impurity(weighted_impurity)

FAIL  weighted_impurity: a perfect split gives 0
      expected: 0.0
      got:      None


### Exercise 4: `predict_with_tree`

A tree can be stored as nested dictionaries. An **internal node** looks like

```python
{"feature": 0, "threshold": 2.5, "left": {...}, "right": {...}}
```

meaning "if `x[feature] <= threshold`, go to `left`, otherwise go to `right`". A **leaf** is `{"value": v}`.

Write `predict_with_tree(tree, X)` that returns a NumPy array with one prediction per row of `X`. `X` may be a 2D array, a list of lists or a DataFrame (use the column positions).

Example:

```python
tree = {"feature": 0, "threshold": 2.5,
        "left": {"value": "small"},
        "right": {"feature": 1, "threshold": 10, "left": {"value": "medium"}, "right": {"value": "large"}}}
predict_with_tree(tree, [[1, 50], [3, 5], [3, 20]])     # array(['small', 'medium', 'large'])
```

In [8]:
def predict_with_tree(tree: dict, X) -> np.ndarray:
    # your code here
    pass

In [9]:
check_predict_with_tree(predict_with_tree)

FAIL  predict_with_tree: the example
      raised AttributeError: 'NoneType' object has no attribute 'tolist'
      expected: ['small', 'medium', 'large']
      hint: go left when x[feature] <= threshold


## Medium

### Exercise 5: `best_split`

Write `best_split(x, y)` that finds the best threshold on **one feature** `x` for class labels `y`, using the weighted Gini impurity. Return `(threshold, impurity)` as two floats.

- The candidate thresholds are the midpoints between consecutive **distinct** sorted values of `x`. Values `<= threshold` go left.
- Return the candidate with the lowest weighted Gini. If several tie, return the **smallest** threshold.
- If `x` has only one distinct value, no split is possible: return `(None, gini of y)`.

Your `weighted_impurity` is available. This matches the root split of `DecisionTreeClassifier(max_depth=1)` trained on the single feature, which the check compares with.

In [10]:
def best_split(x, y):
    # your code here
    pass

In [11]:
check_best_split(best_split)

FAIL  best_split: a small example
      expected: (2.5, 0.0)
      got:      None
      hint: thresholds are midpoints between distinct values


### Exercise 6: `best_split_regression`

Write `best_split_regression(x, y)` for a **numeric** target: find the threshold on one feature that minimises the weighted mean squared error of the two groups, where each group predicts its own mean. Return `(threshold, mse)` as two floats, where

$$\text{mse} = \frac{\text{SSE}_\text{left} + \text{SSE}_\text{right}}{n}, \qquad \text{SSE} = \sum (y_i - \bar{y})^2$$

- Candidates and ties as in Exercise 5: midpoints between distinct sorted values, smallest threshold on ties, `(None, variance of y)` if no split is possible.
- Make it fast: sort once, then use **cumulative sums** of `y` and `y²` to get every candidate's SSE at once, using `SSE = Σy² - (Σy)² / n`. The check uses 20,000 points.

In [12]:
def best_split_regression(x, y):
    # your code here
    pass

In [13]:
check_best_split_regression(best_split_regression)

FAIL  best_split_regression: a small example
      expected: (2.5, 0.0)
      got:      None


### Exercise 7: `bagging_with_oob`

Write `bagging_with_oob(X, y, X_new, n_trees, seed=0)` that bags regression trees and also computes out-of-bag predictions. Return `(new_predictions, oob_predictions)`:

- `new_predictions`: the average prediction of all the trees for each row of `X_new`
- `oob_predictions`: for each training row, the average prediction of the trees that **didn't** see that row; `np.nan` for any row that every tree saw

Follow this procedure exactly, so results can be checked:

```
rng = np.random.default_rng(seed)
repeat n_trees times:
    idx = rng.integers(0, n, n)                                   # a bootstrap sample
    tree = DecisionTreeRegressor(random_state=0).fit(X[idx], y[idx])
    ...
```

`X` and `X_new` are 2D arrays, `y` a 1D array.

In [14]:
from sklearn.tree import DecisionTreeRegressor


def bagging_with_oob(X, y, X_new, n_trees: int, seed: int = 0):
    # your code here
    pass

In [15]:
check_bagging_with_oob(bagging_with_oob)

FAIL  bagging_with_oob: predictions for new data match the reference
      raised TypeError: 'NoneType' object is not subscriptable
      expected: array(shape=(2000,), dtype=float64, [ 3.4335, 3.6334, 0.4364, ..., -0.2363, 0.0689, 4.3...
      hint: idx = rng.integers(0, n, n) for each tree, from one rng


### Exercise 8: `permutation_importance_scores`

Write `permutation_importance_scores(model, X, y, metric, n_repeats=5, seed=0)` for a fitted model and held-out data. It returns a pandas Series indexed by the columns of `X` (a DataFrame), where each value is the average **increase** in the metric when that column is shuffled. `metric(y_true, y_pred)` is an error measure, where lower is better (for example mean squared error).

Follow this procedure exactly:

```
baseline = metric(y, model.predict(X))
rng = np.random.default_rng(seed)
for each column, in order:
    repeat n_repeats times:
        X_shuffled = a copy of X with that column replaced by rng.permutation of its values
        record metric(y, model.predict(X_shuffled)) - baseline
    the column's importance is the average of its records
```

Don't change `X` itself.

In [16]:
def permutation_importance_scores(model, X, y, metric, n_repeats: int = 5, seed: int = 0) -> pd.Series:
    # your code here
    pass

In [17]:
check_permutation_importance_scores(permutation_importance_scores)

FAIL  permutation_importance_scores: matches the reference procedure
      expected: Series shape=(3,)
          strong    16.841428
          weak       1.172075
          noise     -0.000323
      got:      None
      hint: one rng; columns in order; n_repeats shuffles each


### Exercise 9: `partial_dependence_1d`

Write `partial_dependence_1d(model, X, feature, grid)` that computes a **partial dependence** curve: for each value `v` in `grid`, set the column `feature` to `v` for **every** row of `X`, predict, and average the predictions. Return a float array with one value per grid point.

- `X` is a DataFrame and `feature` is a column name.
- Don't change `X` itself.

The check compares with `sklearn.inspection.partial_dependence(..., method="brute")`.

In [18]:
def partial_dependence_1d(model, X, feature: str, grid) -> np.ndarray:
    # your code here
    pass

In [19]:
check_partial_dependence_1d(partial_dependence_1d)

FAIL  partial_dependence_1d: matches scikit-learn for a forest
      expected: array(shape=(15,), dtype=float64, [ 0.4315, 0.7268, 0.8472, ..., 0.763 , 0.401 , -0.0457])
      got:      None


## Hard

### Exercise 10: `build_tree`

Write `build_tree(X, y, max_depth, min_samples_leaf=1)` that grows a classification tree with the Gini impurity and returns it as nested dictionaries, in the format of Exercise 4 (so `predict_with_tree` can use it).

Rules, applied at each node (the root is at depth 0):

- Make a **leaf** if the depth equals `max_depth`, the node is pure, or it has fewer than `2 × min_samples_leaf` examples. A leaf's `"value"` is the most common class (the smallest label if there's a tie), as a plain Python value (use `.item()` on NumPy values).
- Otherwise, try every feature and every midpoint threshold between consecutive distinct values, keeping only splits that leave at least `min_samples_leaf` examples on **each** side. Choose the lowest weighted Gini; on ties, the lowest feature index, then the lowest threshold.
- If no allowed split has a weighted Gini **strictly lower** than the node's own Gini, make a leaf.
- Internal nodes store `"feature"` (an int position), `"threshold"` (a float), `"left"` (values `<=` threshold) and `"right"`.

With continuous features, your tree's predictions should match scikit-learn's `DecisionTreeClassifier` with the same settings.

In [20]:
def build_tree(X, y, max_depth: int, min_samples_leaf: int = 1) -> dict:
    # your code here
    pass

In [21]:
check_build_tree(build_tree)

FAIL  build_tree: a stump on a small example
      expected: {'feature': 0, 'threshold': 2.5, 'left': {'value': 0}, 'right': {'value': 1}}
      got:      None


### Exercise 11: `gradient_boosting_regressor`

Write `gradient_boosting_regressor(X, y, n_estimators, learning_rate, max_depth)` that trains gradient boosting for regression with squared error, and returns a **function** `predict(X_new)` that returns predictions as a float array.

Follow the algorithm from lesson section 9 exactly:

1. Start every prediction at the mean of `y`.
2. For each of `n_estimators` rounds: compute the residuals `y - current predictions`; fit `DecisionTreeRegressor(max_depth=max_depth, random_state=0)` to them; add `learning_rate ×` its predictions to the current predictions.
3. `predict(X_new)` starts from the same mean and adds `learning_rate ×` every tree's predictions.

`X` and `X_new` are 2D arrays.

In [22]:
from sklearn.tree import DecisionTreeRegressor


def gradient_boosting_regressor(X, y, n_estimators: int, learning_rate: float, max_depth: int):
    # your code here
    pass

In [23]:
check_gradient_boosting_regressor(gradient_boosting_regressor)

FAIL  gradient_boosting_regressor: zero trees predict the mean
      raised TypeError: 'NoneType' object is not callable
      expected: array(shape=(3,), dtype=float64, [1.2776, 1.2776, 1.2776])
      hint: start from y.mean()


### Exercise 12: `binary_gradient_boosting`

Write `binary_gradient_boosting(X, y, n_estimators, learning_rate, max_depth)` for **classification** with labels 0 and 1, using the log loss. Return a function `predict_proba(X_new)` that returns the probability of class 1 as a float array.

The model works on the **log-odds** scale, `F`, with probability `p = 1 / (1 + e^{-F})`:

1. Start with `F = log(p̄ / (1 - p̄))` for every example, where `p̄` is the fraction of 1s in `y`.
2. For each round:
   - `p = sigmoid(F)` and residuals `r = y - p` (the negative gradient of the log loss)
   - fit `DecisionTreeRegressor(max_depth=max_depth, random_state=0)` to `r`
   - **replace each leaf's value** with a Newton step: `Σ r / Σ p(1 - p)` over the training examples in that leaf (use `tree.apply(X)` to find each example's leaf)
   - `F += learning_rate ×` the new leaf value of each example's leaf
3. `predict_proba` starts from the same `F`, adds `learning_rate ×` the leaf values for every tree, and applies the sigmoid.

Raise `ValueError` if `y` doesn't contain both classes.

In [24]:
from sklearn.tree import DecisionTreeRegressor


def binary_gradient_boosting(X, y, n_estimators: int, learning_rate: float, max_depth: int):
    # your code here
    pass

In [25]:
check_binary_gradient_boosting(binary_gradient_boosting)

FAIL  binary_gradient_boosting: zero trees predict the base rate
      raised TypeError: 'NoneType' object is not callable
      expected: array(shape=(3,), dtype=float64, [0.438, 0.438, 0.438])
      hint: start from the log-odds of the base rate


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`: a rematch on the California housing data from Module 12, where trees and ensembles take on the best linear model.